# Checkpoint Evaluation & Comparison

Evaluates every saved checkpoint against its test set using the same
core logic as `tcrp/pipelines/evaluate.py`, extended to cover baseline
models (LSTM, TCN, N-BEATS) and adversarial-trained TCRP checkpoints.

**Note on `evaluate.py` compatibility:** `tcrp/pipelines/evaluate.py`
calls `build_loaders(cfg)` with a flat `PipelineConfig`, but `build_loaders`
in `train.py` expects a nested Hydra `DictConfig` (`cfg.datasets.dataset`,
`cfg.trainers.batch_size`, …). This notebook implements a corrected
`build_test_loader` that works directly with `PipelineConfig`, and
uses the same metric computation as `evaluate.py`.

**Outputs**
- Side-by-side MSE / MAE / RMSE comparison table (all checkpoints)
- Bar charts grouped by dataset
- Per-horizon MSE curves for TCRP vs baselines on ETTh1

In [1]:
import sys
sys.path.insert(0, "/workspaces/TCRP")

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

from tcrp.dataset.datasets import DATASET_META, TimeSeriesDataset
from tcrp.dataset.preprocessing import inverse_transform
from tcrp.model.baselines import build_baseline
from tcrp.model.tcrp_forecaster.forecaster import TCRPConfig, TCRPForecaster
from tcrp.model.tcrp_forecaster.components.adversarial import AdversarialTCRPForecaster
from tcrp.pipelines.config import PipelineConfig
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = Path("/workspaces/TCRP/checkpoints")
RESULTS_DIR    = Path("/workspaces/TCRP/results")

print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")

PyTorch 2.10.0+cu128  |  device: cpu


## Helper Functions

These mirror `evaluate.py` but (a) fix the config-compatibility issue
and (b) add support for baseline models.

In [2]:
def build_test_loader(data: dict):
    """Build (test_loader, mean, std) from a plain data-config dict.

    This is the corrected replacement for evaluate.py's build_loaders(),
    which breaks when called with a flat PipelineConfig instead of a
    nested Hydra DictConfig.
    """
    meta = DATASET_META[data["dataset"]]
    path = str(Path(data["data_root"]) / meta["filename"])

    train_ds = TimeSeriesDataset(
        path=path, split="train",
        T=data["T"], H=data["H"],
        normalise=True,
        target_col=data["target_col"],
        univariate=data["univariate"],
    )
    test_ds = TimeSeriesDataset(
        path=path, split="test",
        T=data["T"], H=data["H"],
        normalise=True,
        target_col=data["target_col"],
        univariate=data["univariate"],
    )
    loader = DataLoader(
        test_ds,
        batch_size=data.get("batch_size", 32),
        shuffle=False,
        num_workers=data.get("num_workers", 0),
    )
    return loader, train_ds.mean, train_ds.std


def _infer_baseline_dims(state: dict, model_type: str) -> tuple[int, int]:
    """Infer (hidden, n_layers) from a baseline checkpoint's weight shapes."""
    if model_type == "lstm":
        hidden   = state["lstm.weight_ih_l0"].shape[0] // 4
        n_layers = sum(1 for k in state if k.startswith("lstm.weight_ih_l"))
    elif model_type == "tcn":
        hidden   = state["input_proj.weight"].shape[0]
        n_layers = sum(
            1 for k in state
            if k.startswith("blocks.") and k.endswith("conv1.weight")
        )
    elif model_type == "nbeats":
        hidden         = state["blocks.0.fc.0.weight"].shape[0]
        n_blocks_total = sum(1 for k in state if k.endswith("fc.0.weight"))
        n_layers       = n_blocks_total // 2   # n_blocks_per_stack (n_stacks=2)
    else:
        raise ValueError(f"Unknown baseline model_type: {model_type!r}")
    return hidden, n_layers


def load_model(entry: dict) -> torch.nn.Module:
    """Load a model from a registry entry.

    TCRP models use the entry's TCRPConfig directly.
    Baseline models infer hidden/layer dims from checkpoint weight shapes.
    Adversarial TCRP checkpoints have a 'base.*' key prefix that is stripped.
    """
    model_type = entry["model_type"]
    ckpt_path  = str(CHECKPOINT_DIR / entry["checkpoint"])
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)

    if model_type == "tcrp":
        model = TCRPForecaster(entry["tcrp_cfg"]).to(DEVICE)
        if entry.get("adversarial") or any(k.startswith("base.") for k in state):
            state = {k.removeprefix("base."): v
                     for k, v in state.items()
                     if k.startswith("base.")}
        model.load_state_dict(state)
    else:
        T, H = entry["data"]["T"], entry["data"]["H"]
        hidden, n_layers = _infer_baseline_dims(state, model_type)
        model = build_baseline(
            model_type=model_type, T=T, H=H,
            hidden=hidden, n_layers=n_layers,
        ).to(DEVICE)
        model.load_state_dict(state)

    model.eval()
    return model


def run_inference(model, test_loader, mean, std):
    """Return inverse-transformed (preds, targets) tensors of shape (N, H)."""
    preds, targets = [], []
    with torch.no_grad():
        for x, y in test_loader:
            out = model(x.to(DEVICE))
            preds.append(inverse_transform(out.y_hat.cpu(), mean, std))
            targets.append(inverse_transform(y, mean, std))
    return torch.cat(preds), torch.cat(targets)


def compute_metrics(p: torch.Tensor, t: torch.Tensor) -> dict:
    """MSE / MAE / RMSE plus per-horizon variants (mirrors evaluate.py)."""
    mse  = float(torch.mean((p - t) ** 2))
    mae  = float(torch.mean(torch.abs(p - t)))
    return {
        "test_mse":        round(mse,        6),
        "test_mae":        round(mae,        6),
        "test_rmse":       round(mse ** 0.5, 6),
        "per_horizon_mse": [round(float(v), 6) for v in torch.mean((p - t) ** 2,        dim=0)],
        "per_horizon_mae": [round(float(v), 6) for v in torch.mean(torch.abs(p - t), dim=0)],
    }


def evaluate_checkpoint(entry: dict) -> dict:
    """Full evaluation pipeline for one registry entry.

    Mirrors tcrp/pipelines/evaluate.py:evaluate() and extends it to
    support baseline models and adversarial TCRP checkpoints.
    """
    test_loader, mean, std = build_test_loader(entry["data"])
    model    = load_model(entry)
    n_params = sum(p.numel() for p in model.parameters())
    preds, targets = run_inference(model, test_loader, mean, std)
    metrics  = compute_metrics(preds, targets)

    return {
        "checkpoint":   entry["checkpoint"],
        "display_name": entry["display_name"],
        "dataset":      entry["dataset"],
        "model_type":   entry["model_type"],
        "variant":      entry.get("variant", "standard"),
        "T":            entry["data"]["T"],
        "H":            entry["data"]["H"],
        "n_params":     n_params,
        "test_samples": len(test_loader.dataset),
        **metrics,
    }

print("Helper functions defined.")

Helper functions defined.


## Checkpoint Registry

Each entry maps a checkpoint file to its `PipelineConfig` and metadata.

In [3]:
# ── Shared TCRPConfig defaults (from configs/models/tcrp.yaml) ────────────────
_TCRP_CFG_DEFAULTS = dict(
    L=20, stride=5, k_max=2,
    alpha=5.0, beta=5.0,
    gamma=0.5, gamma_j=2.0, jump_threshold=3.0, train_std=1.0,
    lambda1=0.1, lambda2=1e-4, lambda3=0.01, lambda4=0.1,
    probabilistic=False, adversarial=False, alpha_max=1.0, warmup_epochs=20,
    encoder_in_ch=1, encoder_hidden=64,
    tcn_encoder_n_layers=4, tcn_encoder_kernel_size=3,
    tcn_encoder_use_weight_norm=True,
    lstm_encoder_bidirectional=False, lstm_encoder_dropout=0.0,
    lstm_encoder_pooling="last",
    attention_hidden=32, attention_temp=1.0,
)

# ── Shared dataset / data-loading configs ─────────────────────────────────────
_DATA = {
    "ETTh1":        dict(dataset="ETTh1",        data_root="tcrp/data/raw", univariate=True,  target_col="OT", T=336, H=96),
    "ETTm2":        dict(dataset="ETTm2",         data_root="tcrp/data/raw", univariate=True,  target_col="OT", T=336, H=96),
    "Weather":      dict(dataset="Weather",       data_root="tcrp/data/raw", univariate=True,  target_col=None, T=336, H=96),
    "ExchangeRate": dict(dataset="ExchangeRate",  data_root="tcrp/data/raw", univariate=True,  target_col=None, T=336, H=96),
}

# Per-dataset period lists (from configs/datasets/*.yaml)
_PERIODS = {
    "ETTh1":        [24, 168],
    "ETTm2":        [96, 672],
    "Weather":      [144, 1008],
    "ExchangeRate": [5, 21],
}

def _tcrp_cfg(dataset, **overrides) -> TCRPConfig:
    d = _DATA[dataset]
    return TCRPConfig(
        **{**_TCRP_CFG_DEFAULTS,
           "H": d["H"],
           "d": _TCRP_CFG_DEFAULTS["encoder_hidden"],
           "K": 0,   # recomputed in __post_init__
           "periods": _PERIODS[dataset],
           **overrides}
    )


# ── Registry ──────────────────────────────────────────────────────────────────
# Each entry:
#   tcrp_cfg  — TCRPConfig for TCRP models; None for baselines
#   data      — dict of dataset / DataLoader params
#   cached_json — path to a pre-computed results JSON (optional)
REGISTRY = [
    # ── ETTh1 ─────────────────────────────────────────────────────────────────
    dict(checkpoint="ETTh1_T336_H96_best.pt",
         display_name="TCRP (legacy best)",
         dataset="ETTh1", model_type="tcrp", variant="standard",
         tcrp_cfg=_tcrp_cfg("ETTh1"),
         data=_DATA["ETTh1"],
         cached_json=CHECKPOINT_DIR / "ETTh1_T336_H96_results.json"),
    dict(checkpoint="ETTh1_tcrp_T336_H96_best.pt",
         display_name="TCRP (best val)",
         dataset="ETTh1", model_type="tcrp", variant="standard",
         tcrp_cfg=_tcrp_cfg("ETTh1"),
         data=_DATA["ETTh1"],
         cached_json=CHECKPOINT_DIR / "ETTh1_tcrp_T336_H96_results.json"),
    dict(checkpoint="ETTh1_tcrp_T336_H96_std_best.pt",
         display_name="TCRP (std best)",
         dataset="ETTh1", model_type="tcrp", variant="standard",
         tcrp_cfg=_tcrp_cfg("ETTh1"),
         data=_DATA["ETTh1"],
         cached_json=None),
    dict(checkpoint="ETTh1_lstm_T336_H96_best.pt",
         display_name="LSTM (best val)",
         dataset="ETTh1", model_type="lstm", variant="standard",
         tcrp_cfg=None,
         data=_DATA["ETTh1"],
         cached_json=CHECKPOINT_DIR / "ETTh1_lstm_T336_H96_results.json"),
    dict(checkpoint="ETTh1_lstm_T336_H96_std_best.pt",
         display_name="LSTM (std best)",
         dataset="ETTh1", model_type="lstm", variant="standard",
         tcrp_cfg=None,
         data=_DATA["ETTh1"],
         cached_json=None),
    dict(checkpoint="ETTh1_nbeats_T336_H96_best.pt",
         display_name="N-BEATS (best val)",
         dataset="ETTh1", model_type="nbeats", variant="standard",
         tcrp_cfg=None,
         data=_DATA["ETTh1"],
         cached_json=CHECKPOINT_DIR / "ETTh1_nbeats_T336_H96_results.json"),
    dict(checkpoint="ETTh1_nbeats_T336_H96_std_best.pt",
         display_name="N-BEATS (std best)",
         dataset="ETTh1", model_type="nbeats", variant="standard",
         tcrp_cfg=None,
         data=_DATA["ETTh1"],
         cached_json=None),
    dict(checkpoint="ETTh1_tcn_T336_H96_best.pt",
         display_name="TCN (best val)",
         dataset="ETTh1", model_type="tcn", variant="standard",
         tcrp_cfg=None,
         data=_DATA["ETTh1"],
         cached_json=CHECKPOINT_DIR / "ETTh1_tcn_T336_H96_results.json"),
    dict(checkpoint="ETTh1_tcn_T336_H96_std_best.pt",
         display_name="TCN (std best)",
         dataset="ETTh1", model_type="tcn", variant="standard",
         tcrp_cfg=None,
         data=_DATA["ETTh1"],
         cached_json=None),
    # ── ETTm2 ─────────────────────────────────────────────────────────────────
    dict(checkpoint="ETTm2_tcrp_T336_H96_standard_std_best.pt",
         display_name="TCRP Standard",
         dataset="ETTm2", model_type="tcrp", variant="standard",
         tcrp_cfg=_tcrp_cfg("ETTm2"),
         data=_DATA["ETTm2"],
         cached_json=None),
    dict(checkpoint="ETTm2_tcrp_T336_H96_regularized_std_best.pt",
         display_name="TCRP Regularized",
         dataset="ETTm2", model_type="tcrp", variant="regularized",
         tcrp_cfg=_tcrp_cfg("ETTm2"),
         data=_DATA["ETTm2"],
         cached_json=None),
    dict(checkpoint="ETTm2_tcrp_T336_H96_adversarial_adv_best.pt",
         display_name="TCRP Adversarial",
         dataset="ETTm2", model_type="tcrp", variant="adversarial",
         adversarial=True,
         tcrp_cfg=_tcrp_cfg("ETTm2"),
         data=_DATA["ETTm2"],
         cached_json=None),
    # ── Weather ───────────────────────────────────────────────────────────────
    dict(checkpoint="Weather_tcrp_T336_H96_standard_std_best.pt",
         display_name="TCRP Standard",
         dataset="Weather", model_type="tcrp", variant="standard",
         tcrp_cfg=_tcrp_cfg("Weather"),
         data=_DATA["Weather"],
         cached_json=None),
    dict(checkpoint="Weather_tcrp_T336_H96_adversarial_adv_best.pt",
         display_name="TCRP Adversarial",
         dataset="Weather", model_type="tcrp", variant="adversarial",
         adversarial=True,
         tcrp_cfg=_tcrp_cfg("Weather"),
         data=_DATA["Weather"],
         cached_json=None),
    # ── ExchangeRate ──────────────────────────────────────────────────────────
    dict(checkpoint="ExchangeRate_tcrp_T336_H96_standard_std_best.pt",
         display_name="TCRP Standard",
         dataset="ExchangeRate", model_type="tcrp", variant="standard",
         tcrp_cfg=_tcrp_cfg("ExchangeRate"),
         data=_DATA["ExchangeRate"],
         cached_json=None),
    dict(checkpoint="ExchangeRate_tcrp_T336_H96_regularized_std_best.pt",
         display_name="TCRP Regularized",
         dataset="ExchangeRate", model_type="tcrp", variant="regularized",
         tcrp_cfg=_tcrp_cfg("ExchangeRate"),
         data=_DATA["ExchangeRate"],
         cached_json=None),
    dict(checkpoint="ExchangeRate_tcrp_T336_H96_adversarial_adv_best.pt",
         display_name="TCRP Adversarial",
         dataset="ExchangeRate", model_type="tcrp", variant="adversarial",
         adversarial=True,
         tcrp_cfg=_tcrp_cfg("ExchangeRate"),
         data=_DATA["ExchangeRate"],
         cached_json=None),
]

# Verify all checkpoint files exist
missing = [e["checkpoint"] for e in REGISTRY
           if not (CHECKPOINT_DIR / e["checkpoint"]).exists()]
if missing:
    print(f"WARNING — missing checkpoints: {missing}")
else:
    print(f"{len(REGISTRY)} checkpoints registered, all files present.")

17 checkpoints registered, all files present.


## Run Evaluation

Loads cached JSON results where available, runs the full evaluation pipeline for the rest.

In [7]:
def _try_load_cached(entry: dict) -> dict | None:
    """Return cached metrics if a valid results JSON exists for this checkpoint."""
    json_path = entry.get("cached_json")
    if json_path is None or not Path(json_path).exists():
        return None
    with open(json_path) as f:
        data = json.load(f)

    # Must contain the core metrics
    if not all(k in data for k in ("test_mse", "test_mae", "test_rmse")):
        return None

    return {
        "checkpoint":   entry["checkpoint"],
        "display_name": entry["display_name"],
        "dataset":      entry["dataset"],
        "model_type":   entry["model_type"],
        "variant":      entry.get("variant", "standard"),
        "T":            336,
        "H":            96,
        "n_params":     data.get("n_params", None),
        "test_samples": data.get("test_samples", None),
        "test_mse":     data["test_mse"],
        "test_mae":     data["test_mae"],
        "test_rmse":    data["test_rmse"],
        "per_horizon_mse": data.get("per_horizon_mse", []),
        "per_horizon_mae": data.get("per_horizon_mae", []),
        "_from_cache":  True,
    }


all_results = []
for entry in REGISTRY:
    ckpt = entry["checkpoint"]
    cached = _try_load_cached(entry)
    if cached is not None:
        all_results.append(cached)
        print(f"  [cache]  {ckpt:<60}  MSE={cached['test_mse']:.4f}")
    else:
        print(f"  [eval]   {ckpt:<60}", end="", flush=True)
        res = evaluate_checkpoint(entry)
        all_results.append(res)
        print(f"  MSE={res['test_mse']:.4f}  MAE={res['test_mae']:.4f}")

print(f"\nDone. Evaluated {len(all_results)} checkpoints.")

  [cache]  ETTh1_T336_H96_best.pt                                        MSE=8.4744
  [cache]  ETTh1_tcrp_T336_H96_best.pt                                   MSE=10.0056
  [eval]   ETTh1_tcrp_T336_H96_std_best.pt                             

/workspaces/TCRP/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/workspaces/TCRP/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1787: UserWarning: Input length L=20 is smaller than the stacked receptive field requirement 30. Results may be unreliable.
  return forward_call(*args, **kwargs)


  MSE=8.9448  MAE=2.4068
  [cache]  ETTh1_lstm_T336_H96_best.pt                                   MSE=9.1425
  [eval]   ETTh1_lstm_T336_H96_std_best.pt                               MSE=7.3901  MAE=2.1527
  [cache]  ETTh1_nbeats_T336_H96_best.pt                                 MSE=7.0864
  [eval]   ETTh1_nbeats_T336_H96_std_best.pt                             MSE=7.0864  MAE=2.0405
  [cache]  ETTh1_tcn_T336_H96_best.pt                                    MSE=9.3059
  [eval]   ETTh1_tcn_T336_H96_std_best.pt                                MSE=9.4009  MAE=2.2889
  [eval]   ETTm2_tcrp_T336_H96_standard_std_best.pt                    

/workspaces/TCRP/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/workspaces/TCRP/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1787: UserWarning: Input length L=20 is smaller than the stacked receptive field requirement 30. Results may be unreliable.
  return forward_call(*args, **kwargs)


  MSE=39.8721  MAE=4.7758
  [eval]   ETTm2_tcrp_T336_H96_regularized_std_best.pt                 

...

  MSE=0.0052  MAE=0.0565
  [eval]   ExchangeRate_tcrp_T336_H96_regularized_std_best.pt          

/workspaces/TCRP/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/workspaces/TCRP/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1787: UserWarning: Input length L=20 is smaller than the stacked receptive field requirement 30. Results may be unreliable.
  return forward_call(*args, **kwargs)


  MSE=0.0043  MAE=0.0563
  [eval]   ExchangeRate_tcrp_T336_H96_adversarial_adv_best.pt          

/workspaces/TCRP/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/workspaces/TCRP/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1787: UserWarning: Input length L=20 is smaller than the stacked receptive field requirement 30. Results may be unreliable.
  return forward_call(*args, **kwargs)


  MSE=0.0044  MAE=0.0558

Done. Evaluated 17 checkpoints.


## Comparison Table

MSE / MAE / RMSE for every checkpoint grouped by dataset.

In [8]:
df = pd.DataFrame(all_results)

# Round for display
for col in ("test_mse", "test_mae", "test_rmse"):
    df[col] = df[col].round(4)

# Human-readable model label
df["Model"] = df["model_type"].str.upper() + df["variant"].apply(
    lambda v: f" ({v.capitalize()})" if v != "standard" else ""
)
df["Checkpoint"] = df["checkpoint"]

# Structured table: Dataset × Model  →  MSE / MAE / RMSE
summary = (
    df[["dataset", "display_name", "Checkpoint", "n_params",
        "test_mse", "test_mae", "test_rmse"]]
    .rename(columns={
        "dataset":      "Dataset",
        "display_name": "Model",
        "n_params":     "# Params",
        "test_mse":     "MSE",
        "test_mae":     "MAE",
        "test_rmse":    "RMSE",
    })
    .sort_values(["Dataset", "MSE"])
    .reset_index(drop=True)
)

# ── Styled display ──────────────────────────────────────────────────────────────
def _highlight_best(s):
    """Highlight the minimum value in each Dataset group (lower is better)."""
    is_min = s == s.min()
    return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_min]


styled = (
    summary.style
    .set_caption("Test-set metrics for all checkpoints  (T=336, H=96; bold = best per dataset)")
    .apply(_highlight_best, subset=["MSE"], axis=0)
    .apply(_highlight_best, subset=["MAE"], axis=0)
    .apply(_highlight_best, subset=["RMSE"], axis=0)
    .format({"# Params": "{:,.0f}", "MSE": "{:.4f}", "MAE": "{:.4f}", "RMSE": "{:.4f}"})
    .set_table_styles([
        {"selector": "thead th",
         "props": [("background-color", "#343a40"), ("color", "white"),
                   ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "center")]},
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"),
                   ("text-align", "left"), ("padding-bottom", "6px")]},
    ])
)

display(styled)

,Dataset,Model,Checkpoint,# Params,MSE,MAE,RMSE
0,ETTh1,N-BEATS (best val),ETTh1_nbeats_T336_H96_best.pt,"2,368,032",7.0864,2.0405,2.6620
1,ETTh1,N-BEATS (std best),ETTh1_nbeats_T336_H96_std_best.pt,"2,368,032",7.0864,2.0405,2.6620
2,ETTh1,LSTM (std best),ETTh1_lstm_T336_H96_std_best.pt,"1,342,560",7.3901,2.1527,2.7185
3,ETTh1,TCRP (legacy best),ETTh1_T336_H96_best.pt,"91,412",8.4744,2.3577,2.9111
4,ETTh1,TCRP (std best),ETTh1_tcrp_T336_H96_std_best.pt,"91,412",8.9448,2.4068,2.9908
5,ETTh1,LSTM (best val),ETTh1_lstm_T336_H96_best.pt,"211,552",9.1425,2.3130,3.0237
6,ETTh1,TCN (best val),ETTh1_tcn_T336_H96_best.pt,"106,208",9.3059,2.3128,3.0506
7,ETTh1,TCN (std best),ETTh1_tcn_T336_H96_std_best.pt,"106,208",9.4009,2.2889,3.0661
8,ETTh1,TCRP (best val),ETTh1_tcrp_T336_H96_best.pt,"91,412",10.0056,2.5440,3.1632
9,ETTm2,TCRP Standard,ETTm2_tcrp_T336_H96_standard_std_best.pt,"91,412",39.8721,4.7758,6.3144


## Per-dataset Summary Table

Condensed view: one best checkpoint per (Dataset, Model family).

In [ ]:
# Best checkpoint per (Dataset, model_type) by MSE
best = (
    df.sort_values("test_mse")
    .groupby(["dataset", "model_type"], sort=False)
    .first()
    .reset_index()
    [["dataset", "model_type", "display_name", "checkpoint",
      "test_mse", "test_mae", "test_rmse", "n_params"]]
    .rename(columns={
        "dataset":      "Dataset",
        "model_type":   "Model Type",
        "display_name": "Best Variant",
        "checkpoint":   "Checkpoint File",
        "test_mse":     "MSE",
        "test_mae":     "MAE",
        "test_rmse":    "RMSE",
        "n_params":     "# Params",
    })
    .sort_values(["Dataset", "MSE"])
    .reset_index(drop=True)
)

print("Best checkpoint per model family (by MSE):\n")
display(
    best.style
    .set_caption("Best checkpoint per (Dataset × Model family)")
    .format({"MSE": "{:.4f}", "MAE": "{:.4f}", "RMSE": "{:.4f}",
             "# Params": "{:,.0f}"})
    .set_table_styles([
        {"selector": "thead th",
         "props": [("background-color", "#495057"), ("color", "white")]},
    ])
    .hide(axis="index")
)

## Bar Charts — MSE and MAE by Dataset

In [ ]:
datasets = df["dataset"].unique()
n_ds = len(datasets)

fig, axes = plt.subplots(2, n_ds, figsize=(5 * n_ds, 8), sharey="row")
if n_ds == 1:
    axes = axes.reshape(2, 1)

palette = plt.cm.tab10.colors

for col, ds in enumerate(datasets):
    sub = df[df["dataset"] == ds].sort_values("test_mse")
    labels = [f"{r['display_name']}" for _, r in sub.iterrows()]
    xs = range(len(sub))
    colors = [palette[i % len(palette)] for i in range(len(sub))]

    for row, metric in enumerate(["test_mse", "test_mae"]):
        ax = axes[row, col]
        bars = ax.bar(xs, sub[metric].values, color=colors, edgecolor="white", linewidth=0.5)
        ax.set_xticks(xs)
        ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
        ax.set_title(f"{ds}" if row == 0 else "", fontsize=11, fontweight="bold")
        ax.set_ylabel(metric.replace("test_", "").upper() if col == 0 else "")
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
        ax.grid(axis="y", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

        # Value labels on bars
        for bar, val in zip(bars, sub[metric].values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=7,
            )

fig.suptitle("Test-set MSE and MAE — all checkpoints  (T=336, H=96)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("/workspaces/TCRP/figures/checkpoint_comparison_bars.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figures/checkpoint_comparison_bars.png")

## Per-horizon MSE Curves — ETTh1

Shows how forecast error evolves over the 96-step horizon for each model.

In [ ]:
etth1_rows = [r for r in all_results
              if r["dataset"] == "ETTh1" and r.get("per_horizon_mse")]

if not etth1_rows:
    print("No per-horizon data available (run evaluation without cache to generate it).")
else:
    fig, (ax_mse, ax_mae) = plt.subplots(1, 2, figsize=(13, 4))

    for i, row in enumerate(etth1_rows):
        H = len(row["per_horizon_mse"])
        steps = range(1, H + 1)
        kw = dict(label=row["display_name"], color=palette[i % len(palette)], lw=1.6)
        ax_mse.plot(steps, row["per_horizon_mse"], **kw)
        ax_mae.plot(steps, row["per_horizon_mae"], **kw)

    for ax, title in [(ax_mse, "Per-horizon MSE"), (ax_mae, "Per-horizon MAE")]:
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Forecast step")
        ax.legend(fontsize=8, ncol=2, framealpha=0.5)
        ax.grid(alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("ETTh1 — Per-horizon error (T=336, H=96)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("/workspaces/TCRP/figures/etth1_per_horizon.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → figures/etth1_per_horizon.png")

## Pivot Table — MSE across All Datasets

In [ ]:
# Best-per-family pivot: rows = Model Type, cols = Dataset
pivot = (
    df.sort_values("test_mse")
    .groupby(["model_type", "dataset"])["test_mse"]
    .min()
    .unstack(level="dataset")
    .round(4)
)
pivot.index.name = "Model"

display(
    pivot.style
    .set_caption("Best MSE per (model family × dataset)  — lower is better")
    .highlight_min(axis=0, props="background-color:#d4edda; font-weight:bold")
    .format("{:.4f}", na_rep="—")
    .set_table_styles([
        {"selector": "thead th",
         "props": [("background-color", "#343a40"), ("color", "white"),
                   ("text-align", "center")]},
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold")]},
    ])
)

## Export Results to CSV

In [ ]:
out_csv = RESULTS_DIR / "checkpoint_evaluation_summary.csv"

export_df = df[[
    "dataset", "model_type", "variant", "display_name",
    "checkpoint", "T", "H", "n_params",
    "test_mse", "test_mae", "test_rmse",
]].copy()

export_df.to_csv(out_csv, index=False)
print(f"Results saved to: {out_csv}")
display(export_df)